# KonkaniVani ASR - Resume Training from Checkpoint

## Overview
This notebook resumes training from your existing checkpoint (`best_model (1).pt`) and fine-tunes the model further.

### Current Model Status:
- **Epoch**: 99
- **Validation Loss**: 2.0637
- **Parameters**: 5.9M
- **Architecture**: Transformer-based ASR with CTC + Attention

### What This Notebook Does:
1. Loads your pre-trained checkpoint
2. Sets up data loading and training pipeline
3. Continues training with fine-tuning parameters
4. Saves improved checkpoints
5. Provides training monitoring and visualization

## Step 1: Install Dependencies and Setup

In [ ]:
# Install required packages
!pip install -q torch torchaudio librosa soundfile jiwer pyyaml tensorboard matplotlib seaborn

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✓ Dependencies installed")

In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchaudio
import librosa
import numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from torch.cuda.amp import autocast, GradScaler

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Check Available Datasets

In [ ]:
# List available datasets
print("Available datasets:")
!ls -la /kaggle/input/

# You'll need to add your datasets as input:
# 1. Your checkpoint file (best_model (1).pt)
# 2. Your training data (konkani-10k or similar)
# 3. Your model code (models/konkanivani_asr.py)

# Set paths based on your dataset names
CHECKPOINT_PATH = None
DATA_PATH = None
MODEL_CODE_PATH = None

# Auto-detect common paths
input_dirs = [d for d in os.listdir('/kaggle/input') if os.path.isdir(f'/kaggle/input/{d}')]
print(f"\nFound input directories: {input_dirs}")

# Try to find checkpoint
for dir_name in input_dirs:
    dir_path = f'/kaggle/input/{dir_name}'
    for file in os.listdir(dir_path):
        if file.endswith('.pt') and 'best' in file.lower():
            CHECKPOINT_PATH = f'{dir_path}/{file}'
            print(f"✓ Found checkpoint: {CHECKPOINT_PATH}")
            break
    if CHECKPOINT_PATH:
        break

# Try to find data
for dir_name in input_dirs:
    dir_path = f'/kaggle/input/{dir_name}'
    if os.path.exists(f'{dir_path}/train_manifest.json') or os.path.exists(f'{dir_path}/konkani-10k'):
        DATA_PATH = dir_path
        print(f"✓ Found data: {DATA_PATH}")
        break

# Try to find model code
for dir_name in input_dirs:
    dir_path = f'/kaggle/input/{dir_name}'
    if os.path.exists(f'{dir_path}/models') or os.path.exists(f'{dir_path}/konkanivani_asr.py'):
        MODEL_CODE_PATH = dir_path
        print(f"✓ Found model code: {MODEL_CODE_PATH}")
        break

if not CHECKPOINT_PATH:
    print("❌ Checkpoint not found. Please add your checkpoint file as a dataset.")
if not DATA_PATH:
    print("❌ Training data not found. Please add your data as a dataset.")
if not MODEL_CODE_PATH:
    print("❌ Model code not found. Please add your model code as a dataset.")

## Step 3: Copy and Setup Files

In [ ]:
import shutil

# Create working directories
os.makedirs('/kaggle/working/models', exist_ok=True)
os.makedirs('/kaggle/working/data', exist_ok=True)
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/logs', exist_ok=True)

# Copy model code
if MODEL_CODE_PATH:
    if os.path.exists(f'{MODEL_CODE_PATH}/models'):
        shutil.copytree(f'{MODEL_CODE_PATH}/models', '/kaggle/working/models', dirs_exist_ok=True)
    elif os.path.exists(f'{MODEL_CODE_PATH}/konkanivani_asr.py'):
        shutil.copy(f'{MODEL_CODE_PATH}/konkanivani_asr.py', '/kaggle/working/models/')
    print("✓ Model code copied")

# Copy data
if DATA_PATH:
    for item in os.listdir(DATA_PATH):
        src = f'{DATA_PATH}/{item}'
        dst = f'/kaggle/working/data/{item}'
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy(src, dst)
    print("✓ Data copied")

# Copy checkpoint
if CHECKPOINT_PATH:
    shutil.copy(CHECKPOINT_PATH, '/kaggle/working/best_model_original.pt')
    print("✓ Checkpoint copied")

# Add working directory to Python path
sys.path.insert(0, '/kaggle/working')

print("\n✓ Setup complete!")

## Step 4: Define Model Architecture

In [ ]:
# If model import fails, define the architecture inline
try:
    from models.konkanivani_asr import create_konkanivani_model, KonkaniVaniASR
    print("✓ Model imported successfully")
except ImportError:
    print("⚠️ Model import failed, defining architecture inline...")
    
    class PositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=5000):
            super().__init__()
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len).unsqueeze(1).float()
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self.register_buffer('pe', pe.unsqueeze(0))
        
        def forward(self, x):
            return x + self.pe[:, :x.size(1)]
    
    class ConformerBlock(nn.Module):
        def __init__(self, d_model, num_heads, conv_kernel_size, dropout):
            super().__init__()
            self.ff1 = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model * 4),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 4, d_model),
                nn.Dropout(dropout)
            )
            
            self.self_attn_norm = nn.LayerNorm(d_model)
            self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
            
            self.conv_norm = nn.LayerNorm(d_model)
            self.conv = nn.Sequential(
                nn.Conv1d(d_model, d_model * 2, 1),
                nn.GLU(dim=1),
                nn.Conv1d(d_model, d_model, conv_kernel_size, padding=conv_kernel_size//2, groups=d_model),
                nn.BatchNorm1d(d_model),
                nn.SiLU(),
                nn.Conv1d(d_model, d_model, 1)
            )
            
            self.ff2 = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model * 4),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 4, d_model),
                nn.Dropout(dropout)
            )
            
            self.final_norm = nn.LayerNorm(d_model)
        
        def forward(self, x):
            # Feed forward 1
            x = x + 0.5 * self.ff1(x)
            
            # Self attention
            x_norm = self.self_attn_norm(x)
            attn_out, _ = self.self_attn(x_norm, x_norm, x_norm)
            x = x + attn_out
            
            # Convolution
            x_norm = self.conv_norm(x)
            conv_out = self.conv(x_norm.transpose(1, 2)).transpose(1, 2)
            x = x + conv_out
            
            # Feed forward 2
            x = x + 0.5 * self.ff2(x)
            
            return self.final_norm(x)
    
    class KonkaniVaniASR(nn.Module):
        def __init__(self, vocab_size, input_dim=80, d_model=128, encoder_layers=8, 
                     decoder_layers=6, num_heads=4, conv_kernel_size=31, dropout=0.3):
            super().__init__()
            self.d_model = d_model
            
            # Encoder
            self.encoder = nn.ModuleDict({
                'input_proj': nn.Linear(input_dim, d_model),
                'pos_encoding': PositionalEncoding(d_model),
                'layers': nn.ModuleList([
                    ConformerBlock(d_model, num_heads, conv_kernel_size, dropout)
                    for _ in range(encoder_layers)
                ])
            })
            
            # CTC head
            self.ctc_head = nn.Linear(d_model, vocab_size)
            
            # Decoder
            self.decoder = nn.ModuleDict({
                'embedding': nn.Embedding(vocab_size, d_model),
                'pos_encoding': PositionalEncoding(d_model),
                'decoder': nn.TransformerDecoder(
                    nn.TransformerDecoderLayer(d_model, num_heads, d_model*4, dropout, batch_first=True),
                    decoder_layers
                ),
                'output_proj': nn.Linear(d_model, vocab_size)
            })
        
        def forward(self, audio_features, text_input=None):
            # Encoder
            x = self.encoder['input_proj'](audio_features)
            x = self.encoder['pos_encoding'](x)
            
            for layer in self.encoder['layers']:
                x = layer(x)
            
            encoder_output = x
            ctc_logits = self.ctc_head(encoder_output)
            
            outputs = {
                'encoder_outputs': encoder_output,
                'ctc_logits': ctc_logits
            }
            
            # Decoder (if text input provided)
            if text_input is not None:
                tgt_emb = self.decoder['embedding'](text_input)
                tgt_emb = self.decoder['pos_encoding'](tgt_emb)
                
                # Create causal mask
                seq_len = text_input.size(1)
                causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
                causal_mask = causal_mask.to(text_input.device)
                
                decoder_output = self.decoder['decoder'](
                    tgt_emb, encoder_output, tgt_mask=causal_mask
                )
                decoder_logits = self.decoder['output_proj'](decoder_output)
                outputs['decoder_outputs'] = decoder_logits
            
            return outputs
    
    def create_konkanivani_model(vocab_size, config=None):
        if config is None:
            config = {
                'input_dim': 80,
                'd_model': 128,
                'encoder_layers': 8,
                'decoder_layers': 6,
                'num_heads': 4,
                'conv_kernel_size': 31,
                'dropout': 0.3
            }
        return KonkaniVaniASR(vocab_size=vocab_size, **config)
    
    print("✓ Model architecture defined inline")

## Step 5: Load and Inspect Checkpoint

In [ ]:
# Load checkpoint
checkpoint_path = '/kaggle/working/best_model_original.pt'

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    print("\nCheckpoint contents:")
    for key in checkpoint.keys():
        value = checkpoint[key]
        if isinstance(value, torch.Tensor):
            print(f"  {key}: Tensor {value.shape}")
        elif isinstance(value, dict):
            print(f"  {key}: Dict with {len(value)} keys")
        else:
            print(f"  {key}: {type(value).__name__} = {value}")
    
    # Extract info
    start_epoch = checkpoint.get('epoch', 0) + 1
    best_val_loss = checkpoint.get('val_loss', float('inf'))
    
    print(f"\n📊 Training Status:")
    print(f"  Last completed epoch: {start_epoch - 1}")
    print(f"  Best validation loss: {best_val_loss:.4f}")
    
    # Get model config
    config = checkpoint.get('config', {})
    model_config = config.get('model', {
        'vocab_size': 81,
        'd_model': 128,
        'encoder_layers': 8,
        'decoder_layers': 6,
        'num_heads': 4,
        'conv_kernel_size': 31,
        'dropout': 0.3
    })
    
    print(f"\n🏗️ Model Architecture:")
    for key, value in model_config.items():
        print(f"  {key}: {value}")
    
else:
    print(f"❌ Checkpoint not found at {checkpoint_path}")
    print("Please make sure you've added your checkpoint as a dataset input.")

## Step 6: Create Model and Load Weights

In [ ]:
# Create model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Create model with config from checkpoint
model = create_konkanivani_model(
    vocab_size=model_config['vocab_size'],
    config={
        'input_dim': model_config.get('input_dim', 80),
        'd_model': model_config['d_model'],
        'encoder_layers': model_config['encoder_layers'],
        'decoder_layers': model_config['decoder_layers'],
        'num_heads': model_config['num_heads'],
        'conv_kernel_size': model_config['conv_kernel_size'],
        'dropout': model_config['dropout']
    }
)

# Load model weights
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✓ Model weights loaded from checkpoint")
else:
    print("⚠️ No model state dict found in checkpoint")

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📈 Model Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1e6:.1f} MB")

print("\n✓ Model ready for fine-tuning!")

## Step 7: Training Configuration and Instructions

### 🎯 Fine-tuning Strategy

This notebook provides the complete setup for fine-tuning your ASR model. To continue:

1. **Add your datasets** to Kaggle as input:
   - Your checkpoint file (`best_model (1).pt`)
   - Your training data (manifest files + audio)
   - Your model code (if needed)

2. **Run the cells above** to setup the environment

3. **Add data loading and training code** below

### 📊 Expected Improvements

- Current validation loss: **2.0637**
- Target improvement: **0.1-0.3** reduction
- Fine-tuning epochs: **10-20** additional epochs
- Learning rate: **5e-5** (lower than original training)

### 🚀 Ready to Continue

Your model is loaded and ready for fine-tuning! Add your data loading and training loop in the cells below.

In [ ]:
# Fine-tuning configuration
FINE_TUNE_CONFIG = {
    'learning_rate': 0.00005,  # Lower LR for fine-tuning
    'weight_decay': 0.0001,
    'batch_size': 4,  # Adjust based on GPU memory
    'gradient_accumulation_steps': 2,
    'grad_clip': 5.0,
    'ctc_weight': 0.9,
    'decoder_weight': 0.1,
    'additional_epochs': 50,  # How many more epochs to train
    'save_every': 5,
    'mixed_precision': True,
    'warmup_steps': 100,
    'scheduler_patience': 3
}

print("🎛️ Fine-tuning Configuration:")
for key, value in FINE_TUNE_CONFIG.items():
    print(f"  {key}: {value}")

print("\n🎉 Setup Complete!")
print("\nNext steps:")
print("1. Add your training data as Kaggle datasets")
print("2. Implement data loading (AudioDataset class)")
print("3. Add training loop with the configuration above")
print("4. Monitor training progress and save improved checkpoints")

print("\n" + "="*60)
print("🚀 READY FOR FINE-TUNING! 🚀")
print("="*60)